# Apache Iceberg com Apache Spark — Mercado de Trabalho em IA

## Cenário

Este notebook analisa o mesmo dataset **AI Job Market Insights** (500 registros de vagas em IA) mas usando **Apache Iceberg** em vez de Delta Lake. Demonstra as capacidades **ACID** e **Time Travel via Snapshots**.

> **Pré-requisito:** encerre o kernel do `delta_lake.ipynb` antes de rodar este. Dois SparkSessions em `local[*]` travam.

## Operações Demonstradas

1. **Carga Inicial** — lê o CSV e cria a tabela
2. **INSERT** — adiciona novos registros
3. **UPDATE** — modifica registros existentes
4. **DELETE** — remove registros
5. **Snapshots** — inspeciona o histórico (equivalente a DESCRIBE HISTORY)
6. **Time Travel** — consulta dados em snapshots anteriores

## Configuração Inicial

Inicializa a SparkSession com suporte a Apache Iceberg. Configura o catálogo Hadoop local `local` e limpa artefatos de execuções anteriores.

In [1]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession

project_root = Path.cwd()
data_path = str(project_root / "data" / "ai_job_market_insights.csv")
iceberg_warehouse_dir = project_root / "iceberg-warehouse"

iceberg_table_path = iceberg_warehouse_dir / "db_jobs" / "ai_jobs_iceberg"
if iceberg_table_path.exists():
    shutil.rmtree(iceberg_table_path)

spark = (
    SparkSession.builder
    .appName("Iceberg-AI-Jobs")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", str(iceberg_warehouse_dir.resolve()))
    .getOrCreate()
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db_jobs")
spark.sql("DROP TABLE IF EXISTS local.db_jobs.ai_jobs_iceberg")
spark

26/04/26 15:46:18 WARN Utils: Your hostname, Xande resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/26 15:46:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/alexandre/Apache-spark/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/alexandre/.ivy2/cache
The jars for the packages stored in: /home/alexandre/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-35f307ef-35e4-4106-9812-5e7c8720ffa3;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 88ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apac

## Carga Inicial

Lê o CSV e cria a tabela Iceberg `local.db_jobs.ai_jobs_iceberg` com os 500 registros de vagas em IA.

In [2]:
df = spark.read.option("header", True).option("inferSchema", True).csv(data_path)
df.createOrReplaceTempView("ai_jobs_src")

spark.sql("""
CREATE TABLE local.db_jobs.ai_jobs_iceberg (
    Job_Title             STRING,
    Industry              STRING,
    Company_Size          STRING,
    Location              STRING,
    AI_Adoption_Level     STRING,
    Automation_Risk       STRING,
    Required_Skills       STRING,
    Salary_USD            DOUBLE,
    Remote_Friendly       STRING,
    Job_Growth_Projection STRING
) USING iceberg
""")

spark.sql("INSERT INTO local.db_jobs.ai_jobs_iceberg SELECT * FROM ai_jobs_src")

total = spark.sql("SELECT COUNT(*) FROM local.db_jobs.ai_jobs_iceberg").collect()[0][0]
print(f"✓ Registros carregados: {total}")
spark.sql("SELECT * FROM local.db_jobs.ai_jobs_iceberg LIMIT 3").show(truncate=False)

✓ Registros carregados: 500
+---------------------+-------------+------------+---------+-----------------+---------------+---------------+------------------+---------------+---------------------+
|Job_Title            |Industry     |Company_Size|Location |AI_Adoption_Level|Automation_Risk|Required_Skills|Salary_USD        |Remote_Friendly|Job_Growth_Projection|
+---------------------+-------------+------------+---------+-----------------+---------------+---------------+------------------+---------------+---------------------+
|Cybersecurity Analyst|Entertainment|Small       |Dubai    |Medium           |High           |UX/UI Design   |111392.16524315962|Yes            |Growth               |
|Marketing Specialist |Technology   |Large       |Singapore|Medium           |High           |Marketing      |93792.56246610906 |No             |Decline              |
|AI Researcher        |Technology   |Large       |Singapore|Medium           |High           |UX/UI Design   |107170.26306894995|Yes

## 🔹 ACID: INSERT

Adiciona um novo registro — vaga de Data Engineer em Urussanga, SC (Brasil).

In [3]:
spark.sql("""
INSERT INTO local.db_jobs.ai_jobs_iceberg VALUES
  ('Data Engineer', 'Technology', 'Medium', 'Urussanga',
   'High', 'Low', 'Python,Spark,SQL', 95000.00, 'Yes', 'Growth')
""")

print("✓ INSERT executado. Nova vaga em Urussanga:")
spark.sql("SELECT * FROM local.db_jobs.ai_jobs_iceberg WHERE Location = 'Urussanga'").show(truncate=False)

✓ INSERT executado. Nova vaga em Urussanga:
+-------------+----------+------------+---------+-----------------+---------------+----------------+----------+---------------+---------------------+
|Job_Title    |Industry  |Company_Size|Location |AI_Adoption_Level|Automation_Risk|Required_Skills |Salary_USD|Remote_Friendly|Job_Growth_Projection|
+-------------+----------+------------+---------+-----------------+---------------+----------------+----------+---------------+---------------------+
|Data Engineer|Technology|Medium      |Urussanga|High             |Low            |Python,Spark,SQL|95000.0   |Yes            |Growth               |
+-------------+----------+------------+---------+-----------------+---------------+----------------+----------+---------------+---------------------+



## 🔹 ACID: UPDATE

Reajusta o salário em +10% para todas as vagas com alto nível de adoção de IA E baixo risco de automação.

In [4]:
spark.sql("""
UPDATE local.db_jobs.ai_jobs_iceberg
SET Salary_USD = ROUND(Salary_USD * 1.10, 2)
WHERE AI_Adoption_Level = 'High' AND Automation_Risk = 'Low'
""")

print("✓ UPDATE executado. Amostra de vagas reajustadas:")
spark.sql("""
SELECT Job_Title, Location, AI_Adoption_Level, Automation_Risk, Salary_USD
FROM local.db_jobs.ai_jobs_iceberg
WHERE AI_Adoption_Level = 'High' AND Automation_Risk = 'Low'
LIMIT 4
""").show(truncate=False)

✓ UPDATE executado. Amostra de vagas reajustadas:
+-------------+---------+-----------------+---------------+----------+
|Job_Title    |Location |AI_Adoption_Level|Automation_Risk|Salary_USD|
+-------------+---------+-----------------+---------------+----------+
|AI Researcher|London   |High             |Low            |82517.45  |
|Sales Manager|Singapore|High             |Low            |106518.04 |
|Sales Manager|Dubai    |High             |Low            |91079.29  |
|HR Manager   |Tokyo    |High             |Low            |80655.63  |
+-------------+---------+-----------------+---------------+----------+



## 🔹 ACID: DELETE

Remove todas as vagas com perspectiva de declínio (`Job_Growth_Projection = 'Decline'`).

In [5]:
antes = spark.sql("SELECT COUNT(*) FROM local.db_jobs.ai_jobs_iceberg").collect()[0][0]
spark.sql("DELETE FROM local.db_jobs.ai_jobs_iceberg WHERE Job_Growth_Projection = 'Decline'")
depois = spark.sql("SELECT COUNT(*) FROM local.db_jobs.ai_jobs_iceberg").collect()[0][0]

removidos = antes - depois
print(f"✓ DELETE executado.")
print(f"  Antes: {antes} | Depois: {depois} | Removidos: {removidos}")

✓ DELETE executado.
  Antes: 501 | Depois: 332 | Removidos: 169


## Snapshots

Exibe todas as versões da tabela Iceberg. Cada snapshot é identificado por um ID único e armazena o timestamp e a operação realizada.

In [6]:
spark.sql("SELECT * FROM local.db_jobs.ai_jobs_iceberg.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                       |summary    

## Time Travel

Volta no tempo e consulta a tabela na **versão inicial** imediatamente após a carga inicial (antes de INSERT, UPDATE e DELETE).

In [7]:
first_snapshot = spark.sql("""
    SELECT snapshot_id
    FROM local.db_jobs.ai_jobs_iceberg.snapshots
    ORDER BY committed_at ASC
    LIMIT 1
""").collect()[0][0]

total_snap0 = spark.sql(
    f"SELECT COUNT(*) FROM local.db_jobs.ai_jobs_iceberg VERSION AS OF {first_snapshot}"
).collect()[0][0]

print(f"✓ Time Travel — Snapshot {first_snapshot}")
print(f"  Total de registros: {total_snap0}")
print("\n  Primeiros 3 registros do estado original:")
spark.sql(
    f"SELECT * FROM local.db_jobs.ai_jobs_iceberg VERSION AS OF {first_snapshot} LIMIT 3"
).show(truncate=False)

✓ Time Travel — Snapshot 1401408728893388936
  Total de registros: 500

  Primeiros 3 registros do estado original:
+---------------------+-------------+------------+---------+-----------------+---------------+---------------+------------------+---------------+---------------------+
|Job_Title            |Industry     |Company_Size|Location |AI_Adoption_Level|Automation_Risk|Required_Skills|Salary_USD        |Remote_Friendly|Job_Growth_Projection|
+---------------------+-------------+------------+---------+-----------------+---------------+---------------+------------------+---------------+---------------------+
|Cybersecurity Analyst|Entertainment|Small       |Dubai    |Medium           |High           |UX/UI Design   |111392.16524315962|Yes            |Growth               |
|Marketing Specialist |Technology   |Large       |Singapore|Medium           |High           |Marketing      |93792.56246610906 |No             |Decline              |
|AI Researcher        |Technology   |Large  

In [8]:
spark.stop()
print("✓ SparkSession encerrada.")

✓ SparkSession encerrada.
